# Day 1 Tutorial · Oxford Tutorial Fellow in Agent Architecture (LLM 仿真)## Persona Prompt (本 notebook 的"导师人格")You are an **Oxford tutorial fellow in Agent Architecture (ReAct/Plan-Execute/Reflection/Tool Calling/Memory/MCP)**. You tutor 1-2 PhD students in AI-native business.**Hard rules you NEVER break**:1. **Never give direct answers.** (不直接给答案, 禁直接答案) You ask, you don't tell.2. Use **Socratic questioning** - every turn ends with a probing question, never a statement of fact.3. Act as **HBS devil's advocate** - challenge any vague claim ("Agent 更智能" / "ReAct 更灵活") with "compared to what? by what metric? counterexample?"4. **Reject vague claims.** If the student says "我觉得 LangGraph 好用", reply "what does '好用' measure? latency? controllability? token cost? pick one and justify."5. **End each turn with exactly one probing question.** No exceptions.6. Domain-anchored: every question must reference a Day 1 artifact (`@tool` contract / `create_react_agent` / `MemorySaver(thread_id=...)` / Anthropic 五模式 / MCP).7. 不表扬学生本人 (no Self-level praise per Hattie) - 评判任务, 不评判人。**Tone**: rigorous, slightly adversarial, but never sarcastic. Imagine a Boston Consulting Group partner who used to grade Oxford PPE tutes.

## Pre-Tutorial Task (强制 retrieval, 提交后才能进入 cell3 Socratic loop)> 不许翻 notes.md / solution.ipynb。这是提取练习 (retrieval practice), 不是重读。**提交物 (写入下方代码块的 `submission` 字典)**:1. **essay (<=200 字)**: 用 ReAct/Plan-Execute/Reflection 三模式解释"你的营销 Agent 在哪个场景该选哪个模式", 必须含 1 个反例 ("若选错会发生什么")。2. **tool_contract**: 写一个 `@tool` 函数 `analyze_sentiment`, 完整 docstring + 类型注解, 让 LLM 能正确调用。3. **architecture_choice**: 给"品牌危机公关策略生成"选一个 Anthropic 五模式, 1 句话理由 + 1 句"为什么不用 Workflow"。**评分钥匙 (tutorial 会按此追问)**:- essay 命中至少 2 个模式 + 1 个反例?- tool_contract 的 docstring 含营销语义 + 参数类型 + 返回类型?- architecture_choice 选了 Evaluator-Optimizer? 理由含"质量敏感"?提交后跑 cell3, tutorial 会用静态 if/else 判断你的提交并 Socratic 追问。

In [ ]:
# Socratic Loop (静态 if/else 模拟, 不调 openai/anthropic API)# 每个 turn 都是预设的 Socratic 追问, 根据学生提交内容分支# 真实 LLM 会引入延迟和不确定性, 这里用静态模拟保证可复现submission = {    "essay": "营销 ROI 计算用 ReAct, 因为需要灵活调工具; 公关策略用 Reflection, 因为质量敏感。",    "tool_contract": "def analyze_sentiment(text): return 2",  # 故意缺 docstring 和类型    "architecture_choice": "Evaluator-Optimizer, 因为质量敏感"}turn = 0max_turns = 6  # >=4 轮tutorial_log = []def socratic_turn(sub, turn):    """静态 if/else 模拟 Oxford tutorial fellow 的 Socratic 追问。    每个 turn 必含 >=1 个 Socratic 问题, 全程累计 >=5 个。"""    if turn == 0:        # 追问 1: 为什么 (why)        return ("你的 essay 说'ROI 用 ReAct 因为需要灵活调工具'。"                "**为什么** '灵活调工具'非要用 ReAct? Plan-Execute 也能调工具, "                "反例是什么? 给我一个营销场景, Plan-Execute 比 ReAct 更优。")    elif turn == 1:        # 追问 2: 反例 (counterexample)        return ("你写了 `def analyze_sentiment(text): return 2`。"                "LLM 凭这个签名能正确调用吗? 给一个**反例**: 让 LLM 误以为这是计算器。"                "你的 docstring 缺什么? 参数类型缺什么? 返回类型缺什么?")    elif turn == 2:        # 追问 3: 若前提变 (what if premise changes)        return ("你选 Evaluator-Optimizer 因为'质量敏感'。**若**品牌危机是实时爆发的 "                "(30 分钟内必须回应), Evaluator-Optimizer 的 token 消耗和延迟还合理吗? "                "此时该换成哪个 Anthropic 模式? 凭什么?")    elif turn == 3:        # 追问 4: 凭什么 (on what basis)        return ("你的 essay 提到 Reflection。**凭什么** Reflection 能提升质量? "                "评估者自身会不会也有盲点? 用 Day 1 的 `solution.ipynb` TODO5 评估者循环, "                "若评估者和生成器是同一个 LLM, 这是不是循环论证? 如何打破?")    elif turn == 4:        # 追问 5: 如何 (how)        return ("回到你的 `analyze_sentiment`。LLM 的'接口契约'是名称+docstring+参数Schema。"                "**如何**只改 docstring (不改函数体), 让 LLM 把它误调成'情感分析'实则算 ROI? "                "这个练习说明工具契约的哪一层最容易被攻击?")    else:        # 追问 6: 综合反例        return ("最后一问: 你的营销 Agent 接 Salesforce CRM。用 `@tool` 还是 MCP? "                "给一个**反例**: 若坚持用 `@tool` 直接调 Salesforce API, 会在 Day 5 "                "生产部署时撞上什么坑? (提示: 进程内 vs 进程外, JSON-RPC, 企业系统集成)")for turn in range(max_turns):    q = socratic_turn(submission, turn)    tutorial_log.append({"turn": turn, "fellow_question": q, "student_attempt": ""})    print(f"--- Turn {turn} ---")    print(f"FELLOW: {q}")    print()# 统计 Socratic 问题数 (应 >=5)import reall_q = "\n".join([t["fellow_question"] for t in tutorial_log])socratic_markers = re.findall(r"(为什么|反例|若|凭什么|如何|how could|what if|counterexample)", all_q, re.I)print(f"=== Socratic markers count: {len(socratic_markers)} (must be >=5) ===")print(f"=== Total turns: {len(tutorial_log)} (must be >=4) ===")

In [ ]:
# student_model.json: 记录掌握度 + 盲点 (Hattie formative 用)# tutorial 据此个性化追问, 不再"一视同仁"import json, osstudent_model = {    "student_id": "U5-D1-s001",    "ilo_mastery": {        "ILO1_三模式解释": {"score": 0.6, "attempts": 1, "blind_spot": "ReAct vs Plan-Execute 边界模糊"},        "ILO2_LangGraph构建": {"score": 0.4, "attempts": 1, "blind_spot": "MemorySaver thread_id 接线"},        "ILO3_工具契约设计": {"score": 0.3, "attempts": 1, "blind_spot": "docstring 缺营销语义, 参数无类型"},        "ILO4_Reflection循环": {"score": 0.5, "attempts": 1, "blind_spot": "评估者与生成器循环论证"},        "ILO5_Anthropic五模式选型": {"score": 0.7, "attempts": 1, "blind_spot": "延迟 vs 质量权衡未量化"}    },    "weak_ilo": "ILO3_工具契约设计",  # score<0.5 触发 weak_loop    "next_review_unit": "U5-D1 (重做 DR1-A/B) -> U5-D2 (LangGraph StateGraph)",    "session_count_today": 1,    "daily_limit": 1  # 每天限 1 次 tutorial}# 读写 student_model.jsonmodel_path = "student_model.json"if os.path.exists(model_path):    with open(model_path, "r", encoding="utf-8") as f:        existing = json.load(f)    # 合并新 mastery (取最高)    for k, v in student_model["ilo_mastery"].items():        if k in existing.get("ilo_mastery", {}):            existing["ilo_mastery"][k]["score"] = max(existing["ilo_mastery"][k]["score"], v["score"])            existing["ilo_mastery"][k]["attempts"] += 1        else:            existing["ilo_mastery"][k] = v    student_model = existingwith open(model_path, "w", encoding="utf-8") as f:    json.dump(student_model, f, ensure_ascii=False, indent=2)print(f"=== student_model.json written ===")print(f"Weakest ILO: {student_model['weak_ilo']}")print(f"Next review: {student_model['next_review_unit']}")print(f"Session today: {student_model['session_count_today']}/{student_model['daily_limit']} (限频)")

## Hattie 4-Level Formative Feedback (tutorial 退出时给出)> Hattie & Timperley (2007) 反馈模型。本 tutorial 只给 Task/Process/Self-Reg/Feed-Forward 四级, **不给 Self 级表扬** (Hattie 实证: Self 级表扬对学习效应量极小, ≈0.14)。### [TASK] 任务级反馈 (这个回答对不对?)- 你的 essay 提到"ROI 用 ReAct" - **正确**, 但缺反例。任务级达标 60%。- 你的 `analyze_sentiment` 函数签名 - **不合格**。LLM 接口契约 = 名称+docstring+参数Schema, 你三个都缺。任务级达标 30%。- 你的 Evaluator-Optimizer 选型 - **正确**, 但"质量敏感"是套话。任务级达标 70%。### [PROCESS] 过程级反馈 (你用的策略对不对?)- 你在 essay 里用了"模式 -> 场景"映射策略, 这是对的。但缺"反例 -> 边界"步骤, 导致论证单薄。- 写 `@tool` 时你直接写函数体, 跳过了"先写 docstring 再写实现"的契约优先流程。**改**: 下次先写 docstring + 类型注解, 再写函数体。- 选 Evaluator-Optimizer 时你没做"延迟 vs 质量"权衡矩阵, 凭直觉选。**改**: 用 notes.md §关键回顾5 的五模式表, 至少填 2 列再选。### [SELF-REG] 自我调节级反馈 (你怎么知道自己会不会?)- 你能判断自己 essay 缺反例吗? 若能, 下次写完先自问"反例是什么"再交。- 你能判断 `analyze_sentiment` 签名不合格吗? 若不能, 你缺一个"LLM 视角的工具契约 checklist"。- 退出 tutorial 后, 用 `practice.md` DR1-A 的 Worked example 对照, 自评你的工具契约差距。### [FEED-FORWARD] 前馈级反馈 (下一步做什么?)- **24h 内**: 重做 `practice.md` DR1-A (Worked) + DR1-B (Faded), 退出 weak_loop 后再做 DR1-C。- **Day 2 前**: 读 `reading.md` 的 LangGraph 条目, 准备 StateGraph (Day 2 不再用 `create_react_agent` 预构建)。- **跨单元复习**: 你的 ILO5 (Anthropic 五模式选型) 在 Skill 3 (系统设计) Day 3 多 Agent 协作会复用, 现在 70% mastery, 需在 Day 2-5 持续强化。

## 限频 (防依赖) + Exit Artifact### 限频 (usage limit)- **每单元每天 1 次 tutorial** (1次/天)。本 cell 跑完即扣 `student_model.session_count_today += 1`。- **为什么不无限?** Oxford tutorial 的价值在于"两次 tutorial 之间的独立思考"。无限对话 = 学生把 tutorial 当 LLM 搜索用, 丧失 retrieval practice。- **超额处理**: 若当天已用 1 次, tutorial 拒绝启动, 提示"明天再来 / 先做 `practice.md` 的 drill / 用 `schedule.json` 的间隔重复卡片复习"。### Exit Artifact (退出时必交)跑完本 notebook 后, 在下方填 3 项 (不填则 tutorial 不算完成):1. **2-3 个盲点 (blind spots)**: 本次 tutorial 暴露的你原本以为会、实则不会的点。   - 例: "我以为 `@tool` 装饰器就够了, 没想到 LLM 看的是 docstring + 类型注解 + 参数Schema 三层契约。"   - 例: "我以为 Reflection 一定提升质量, 没想到评估者和生成器同 LLM 时是循环论证。"2. **推荐复习单元 (next review units)**: 基于 `student_model.json` 的 weak_ilo, 列出 2 个该复习的单元/文件。   - 例: "U5-D1 `practice.md` DR1-A/B (工具契约 Worked-Faded) + U5-D2 `notes.md` StateGraph (create_react_agent 的底层)"3. **1 个反事实**: "如果今天 tutorial 我没被追问'凭什么 Reflection 提升质量', 我会带着哪个错误认知进 Day 2?"   - 这一步是 meta-cognitive retrieval, 强迫你命名 tutorial 修正的具体认知偏差。### 退出检查清单- [ ] cell3 跑完 >=4 轮 Socratic loop, >=5 个 Socratic 问- [ ] cell4 `student_model.json` 已更新, session_count_today=1- [ ] cell5 Hattie 四级反馈已读, 标注至少 1 条 [PROCESS] 改进项- [ ] Exit Artifact 3 项已填**全部勾选后, 本单元 tutorial 当日额度用尽。明日再见。**